# ConvNeXt로 PCB 결함 찾기 — 데이터 준비 · 전처리 · 학습 · test까지 한 번에

이 노트북 하나만 있으면 **위에서부터 끝까지 실행(Run All)** 해서 test 결과(혼동행렬)까지 볼 수 있다. 다른 파일(`visa.py` 등)은 필요 없다.

| 단계 | 내용 | 시간 (GPU 기준, 대략) |
|---|---|---|
| 0. 데이터 | VisA 데이터셋의 pcb1~pcb4 (없으면 자동으로 내려받음, 약 1.9GB) | 다운로드 속도에 따라 5~20분 (처음 한 번만) |
| 1. split | (pcb 종류, 정상/결함)마다 따로 섞어서 train : val : test = 70 : 15 : 15 | 몇 초 |
| 2. 전처리 | 보드 자르기 → 기준 사진에 맞춰 정렬 → 색·대비 보정(CLAHE) | 5~10분 (처음 한 번만, 이후 저장된 사진 재사용) |
| 3. 학습 | ConvNeXt-Tiny (ImageNet 사전학습) 이진 분류, val AP로 가장 좋은 epoch 저장 | 10~20분 |
| 4. test | val에서 정한 임계값으로 test를 한 번만 평가 → 혼동행렬 | 1분 |

**재현성**: split(seed 42)과 전처리 기준(train 정상 사진으로만 정함)은 우리 조 결과와 똑같이 만들어진다. 학습은 GPU 연산 특성상 실행할 때마다 조금씩 달라서, test 숫자는 우리 결과(맨 아래 표)와 **비슷하게** 나온다.

**데이터 위치**: 노트북 폴더, 한 단계 위, 두 단계 위에 `VisA_20220922` 폴더(안에 pcb1~pcb4)가 있으면 자동으로 찾아서 쓴다. 다른 곳에 있으면 설정 셀의 `DATA_ROOT`에 경로를 적는다. 어디에도 없을 때만 내려받는다.

**필요한 것**: Python 3.10+, NVIDIA GPU 권장 (CPU도 되지만 학습이 매우 느림), 인터넷 (데이터·사전학습 가중치 다운로드)

**데이터 출처**: VisA (Zou et al., ECCV 2022), CC BY 4.0 — https://github.com/amazon-science/spot-diff

In [ ]:
# 처음 한 번만: 필요한 패키지 설치 (이미 있으면 건너뛰어도 됨). 주석(#)을 지우고 실행
# GPU용 torch는 https://pytorch.org 에서 자기 CUDA 버전에 맞는 명령으로 설치하는 것을 추천
# %pip install torch torchvision timm opencv-python scikit-learn pandas matplotlib pillow

## 설정

In [ ]:
import csv, json, os, random, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image

# ===== 여기만 필요하면 바꾸기 =====
DATA_ROOT = None                       # VisA 폴더 (안에 pcb1~pcb4 폴더). None이면 노트북 폴더 → 한 단계 위 → 두 단계 위에서 자동으로 찾음
AUTO_DOWNLOAD = True                   # 데이터가 없으면 자동으로 내려받기
OUT = Path("convnext_outputs")         # 결과 저장 폴더 (split, 전처리 사진, 모델, 그림)
LIMIT = None                           # 빠른 확인용: 숫자를 넣으면 (pcb, 정상/결함)마다 그 수만큼만 사용
EPOCHS = 20                            # 최대 epoch (val AP가 5 epoch 동안 안 오르면 일찍 멈춤)
# =================================

OBJECTS = ["pcb1", "pcb2", "pcb3", "pcb4"]
SEED = 42
RATIOS = (0.7, 0.15, 0.15)             # train / val / test
PREP = {"long_side": 768, "n_samples": 50, "bg_thr": 25.0, "crop_margin": 0.08, "align_scale": 0.25,
        "align_pad": 0.12, "align_max_shift": 0.15, "clahe_clip": 2.0, "clahe_tile": 8}
CNX = {"arch": "convnext_tiny.fb_in22k_ft_in1k", "img_size": 384, "batch_size": 16, "epochs": EPOCHS,
       "lr": 1e-4, "weight_decay": 0.05, "patience": 5}
NUM_WORKERS = 0 if os.name == "nt" else 4   # 윈도우 노트북에서는 0이어야 함 (노트북 안에서 만든 Dataset은 worker로 못 넘김)
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def setup_korean_font():
    from matplotlib import font_manager
    names = {f.name for f in font_manager.fontManager.ttflist}
    for n in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if n in names:
            plt.rcParams["font.family"] = n
            break
    plt.rcParams["axes.unicode_minus"] = False


setup_korean_font()
set_seed()
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT.mkdir(parents=True, exist_ok=True)
PREP_DIR = OUT / "prep_crop_align_clahe"
print("장치:", dev, "| 결과:", OUT.resolve())
if dev.type == "cpu":
    print("⚠️ GPU가 없어서 CPU로 학습함 → 매우 느림 (LIMIT으로 줄여서 확인 추천)")

## 0) 데이터 (없으면 자동 다운로드)

In [ ]:
# 0) 데이터 준비: 이미 있는 VisA 폴더를 찾고, 어디에도 없을 때만 VisA 원본(약 1.9GB)을 받아서 pcb1~pcb4만 꺼냄 (압축 파일은 저장 안 함)
VISA_URL = "https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar"


def missing_files(root):
    """root에 pcb1~4의 image_anno.csv와 거기 적힌 사진이 다 있는지 → 없는 파일 수 (csv가 없으면 None)."""
    root = Path(root)
    if not all((root / o / "image_anno.csv").exists() for o in OBJECTS):
        return None
    n = 0
    for o in OBJECTS:
        with open(root / o / "image_anno.csv", newline="", encoding="utf-8") as f:
            n += sum(not (root / r["image"]).exists() for r in csv.DictReader(f))
    return n


def find_data_root():
    """노트북 폴더 → 한 단계 위 → 두 단계 위 순서로 VisA 폴더를 찾음 (폴더 이름이 달라도 pcb1~4가 바로 들어 있으면 인정)."""
    cands = [Path(DATA_ROOT)] if DATA_ROOT else [
        Path(up) / name for up in [".", "..", "../.."] for name in ["VisA_20220922", "VisA", "visa"]] + [Path("."), Path("..")]
    for c in cands:
        miss = missing_files(c)
        if miss == 0:
            return c, cands
        if miss:
            print(f"⚠️ {c.resolve()}: pcb 폴더는 있는데 사진 {miss}장이 없음 (다운로드가 중간에 멈췄던 것 같음) → 건너뜀")
    return None, cands
    import re, shutil, tarfile, urllib.request
    keep = re.compile(r"^(?:\./)?(pcb[1-4])/(.+)$")
    t0, n = time.time(), 0
    with urllib.request.urlopen(VISA_URL) as resp, tarfile.open(fileobj=resp, mode="r|") as tar:
        for m in tar:
            k = keep.match(m.name)
            if not (k and m.isfile()) or ".." in Path(k.group(2)).parts:
                continue
            dst = DATA_ROOT / k.group(1) / k.group(2)
            dst.parent.mkdir(parents=True, exist_ok=True)
            with tar.extractfile(m) as f, open(dst, "wb") as g:
                shutil.copyfileobj(f, g)
            n += 1
            if n % 2000 == 0:
                print(f"  {n}개 파일 ({time.time()-t0:.0f}s)")
    print(f"완료: {n}개 파일 → {DATA_ROOT.resolve()} ({(time.time()-t0)/60:.1f}분)")


found, cands = find_data_root()
if found is not None:
    DATA_ROOT = found
    print("데이터 있음 → 다운로드 안 함:", DATA_ROOT.resolve())
elif AUTO_DOWNLOAD:
    DATA_ROOT = Path(DATA_ROOT or "VisA_20220922")
    print("찾아본 곳:", [str(c.resolve()) for c in cands])
    print(f"어디에도 없어서 {DATA_ROOT.resolve()}에 내려받는 중... (pcb 폴더가 압축 파일 뒤쪽에 있어서 전체를 한 번 훑음)")
    print("이미 데이터가 있다면 이 셀을 멈추고 설정 셀의 DATA_ROOT에 그 폴더 경로를 적으세요")
    download_pcbs()
    assert missing_files(DATA_ROOT) == 0, "다운로드 후에도 사진이 모자람 → 다시 실행"
else:
    raise FileNotFoundError(f"VisA pcb1~pcb4를 못 찾음 (찾아본 곳: {[str(c) for c in cands]}) → DATA_ROOT에 경로를 적거나 AUTO_DOWNLOAD = True")

## 1) split — train 70% / val 15% / test 15%

In [ ]:
# 1) split: (pcb 종류, 정상/결함) 묶음마다 따로 섞어서 70:15:15 (seed 42 → 언제 돌려도 같은 split)
def list_samples():
    rows = []
    for obj in OBJECTS:
        with open(DATA_ROOT / obj / "image_anno.csv", newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f):
                defect = r["label"].strip()
                rows.append(dict(object=obj, image=r["image"], label=0 if defect == "normal" else 1,
                                 defect=defect, mask=(r.get("mask") or "").strip()))
    return rows


def make_split(samples, ratios=RATIOS, seed=SEED):
    rng = random.Random(seed)
    groups = {}
    for s in samples:
        groups.setdefault((s["object"], s["label"]), []).append(s)
    out = []
    for key in sorted(groups):
        g = sorted(groups[key], key=lambda s: s["image"])
        rng.shuffle(g)
        n = len(g)
        n_val, n_test = round(n * ratios[1]), round(n * ratios[2])
        n_train = n - n_val - n_test
        for i, s in enumerate(g):
            out.append({**s, "split": "train" if i < n_train else ("val" if i < n_train + n_val else "test")})
    return out


def limit_rows(rows, n, seed=SEED):
    if not n:
        return rows
    rng, out, groups = random.Random(seed), [], {}
    for r in rows:
        groups.setdefault((r["object"], r["label"]), []).append(r)
    for k in sorted(groups):
        g = list(groups[k]); rng.shuffle(g); out += g[:n]
    return out


ALL = make_split(list_samples())
pd.DataFrame(ALL).to_csv(OUT / "split.csv", index=False)
split = {s: limit_rows([r for r in ALL if r["split"] == s], LIMIT) for s in ["train", "val", "test"]}
tab = pd.DataFrame(ALL).groupby(["split", "label"]).size().unstack().rename(columns={0: "정상", 1: "결함"})
display(tab.loc[["train", "val", "test"]])
print("이번에 쓰는 사진:", {s: len(r) for s, r in split.items()}, "(LIMIT =", LIMIT, ")")

## 2) 전처리 — 자르기 → 정렬 → 색·대비 보정

In [ ]:
# 2-1) 전처리 함수들: 보드 자르기(crop) → 기준 사진에 맞춰 정렬(align) → 색·대비 보정(CLAHE)
def read_rgb(rel):
    return np.array(Image.open(DATA_ROOT / rel).convert("RGB"))


def board_box(im, bg_thr):
    """사진 테두리 색(=배경)과 다른 가장 큰 덩어리 = 보드. (x0, y0, x1, y1)"""
    lab = cv2.cvtColor(im, cv2.COLOR_RGB2LAB).astype(np.float32)
    b = 5
    border = np.concatenate([lab[:b].reshape(-1, 3), lab[-b:].reshape(-1, 3), lab[:, :b].reshape(-1, 3), lab[:, -b:].reshape(-1, 3)])
    m = (np.linalg.norm(lab - np.median(border, 0), axis=2) > bg_thr).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(m)
    if n <= 1:
        return 0, 0, im.shape[1], im.shape[0]
    i = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    x, y, w, h = stats[i, :4]
    return int(x), int(y), int(x + w), int(y + h)


def find_board(im, bg_thr):
    sc = 0.25   # 1/4로 줄여서 찾고 원래 좌표로
    x0, y0, x1, y1 = board_box(cv2.resize(im, None, fx=sc, fy=sc, interpolation=cv2.INTER_AREA), bg_thr)
    return int(x0 / sc), int(y0 / sc), int(x1 / sc), int(y1 / sc)


def crop_window(box, p, scale=1.0):
    """보드 중심에 (crop_wh × scale) 크기의 창. 사진 밖으로 나가면 안쪽으로 밀어 넣음."""
    W, H = p["orig_size"]
    cw, ch = min(W, int(p["crop_wh"][0] * scale)), min(H, int(p["crop_wh"][1] * scale))
    cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
    x0 = int(np.clip(cx - cw / 2, 0, W - cw)); y0 = int(np.clip(cy - ch / 2, 0, H - ch))
    return x0, y0, x0 + cw, y0 + ch


def cut(im, p, box):
    """정렬용으로 여유 있게 큰 창을 잘라 크기 조정 → (큰 사진, 최종 창 위치)."""
    big, fin = crop_window(box, p, 1 + 2 * p["align_pad"]), crop_window(box, p)
    k = p["size"][0] / p["crop_wh"][0]
    bw, bh = int(round((big[2] - big[0]) * k)), int(round((big[3] - big[1]) * k))
    out = cv2.resize(im[big[1]:big[3], big[0]:big[2]], (bw, bh), interpolation=cv2.INTER_AREA)
    ox = int(np.clip(round((fin[0] - big[0]) * k), 0, bw - p["size"][0]))
    oy = int(np.clip(round((fin[1] - big[1]) * k), 0, bh - p["size"][1]))
    return out, (ox, oy)


def final(im, off, p):
    return im[off[1]:off[1] + p["size"][1], off[0]:off[0] + p["size"][0]]


_TEMPLATES = {}


def template_big(p):
    if p["template"] not in _TEMPLATES:
        im = read_rgb(p["template"])
        _TEMPLATES[p["template"]] = cut(im, p, find_board(im, p["bg_thr"]))
    return _TEMPLATES[p["template"]]


def fit_prep(train_rows, cfg, seed=SEED):
    """pcb마다 train 정상 사진 50장으로: 자를 크기(보드 크기 90번째 백분위 + 8%), 기준 사진(평균에 가장 가까운 한 장)."""
    rng, params = random.Random(seed), {}
    for obj in sorted({r["object"] for r in train_rows}):
        normals = sorted([r for r in train_rows if r["object"] == obj and r["label"] == 0], key=lambda r: r["image"])
        sample = rng.sample(normals, min(cfg["n_samples"], len(normals)))
        W, H = Image.open(DATA_ROOT / sample[0]["image"]).size
        ims = [read_rgb(r["image"]) for r in sample]
        boxes = np.array([find_board(im, cfg["bg_thr"]) for im in ims], dtype=float)
        bw, bh = np.percentile(boxes[:, 2] - boxes[:, 0], 90), np.percentile(boxes[:, 3] - boxes[:, 1], 90)
        cw, ch = int(min(W, bw * (1 + 2 * cfg["crop_margin"]))), int(min(H, bh * (1 + 2 * cfg["crop_margin"])))
        k = cfg["long_side"] / max(cw, ch)
        p = {"crop_wh": [cw, ch], "size": [int(round(cw * k)), int(round(ch * k))], "orig_size": [W, H],
             "bg_thr": cfg["bg_thr"], "align_pad": cfg["align_pad"]}
        crops = np.stack([final(*cut(im, p, b), p).astype(np.float32) for im, b in zip(ims, boxes)])
        mean = crops.mean(0)
        p["template"] = sample[int(np.argmin([np.abs(c - mean).mean() for c in crops]))]["image"]
        params[obj] = p
        print(f"[{obj}] 원본 {W}x{H} → 자를 크기 {cw}x{ch} → 저장 {p['size'][0]}x{p['size'][1]} | 기준 사진 {p['template']}")
    return params


def align_to(im, tmpl, cfg):
    """ECC로 이동+회전을 찾아 기준 사진에 맞춤. 15% 넘게 밀리거나 10도 넘게 돌면 실패로 보고 그대로 둠."""
    s = cfg["align_scale"]
    g = lambda x: cv2.cvtColor(cv2.resize(x, None, fx=s, fy=s, interpolation=cv2.INTER_AREA), cv2.COLOR_RGB2GRAY).astype(np.float32)
    if im.shape != tmpl.shape:
        im = cv2.resize(im, (tmpl.shape[1], tmpl.shape[0]), interpolation=cv2.INTER_AREA)
    warp, ok = np.eye(2, 3, dtype=np.float32), True
    try:
        _, warp = cv2.findTransformECC(g(tmpl), g(im), warp, cv2.MOTION_EUCLIDEAN,
                                       (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 100, 1e-5), None, 5)
        warp[:, 2] /= s
        lim = cfg["align_max_shift"]
        if abs(warp[0, 2]) > lim * im.shape[1] or abs(warp[1, 2]) > lim * im.shape[0] or abs(np.degrees(np.arctan2(warp[1, 0], warp[0, 0]))) > 10:
            ok = False
    except cv2.error:
        ok = False
    if not ok:
        warp = np.eye(2, 3, dtype=np.float32)
    out = cv2.warpAffine(im, warp, (im.shape[1], im.shape[0]), flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP, borderMode=cv2.BORDER_REPLICATE)
    return out, ok


def clahe_color(im, tmpl, cfg):
    """화이트밸런스(R·G·B 평균을 기준 사진에 맞춤) → CLAHE(밝기 채널만)."""
    gain = tmpl.reshape(-1, 3).mean(0) / np.clip(im.reshape(-1, 3).mean(0), 1, None)
    im = np.clip(im.astype(np.float32) * gain, 0, 255).astype(np.uint8)
    lab = cv2.cvtColor(im, cv2.COLOR_RGB2LAB)
    t = int(cfg["clahe_tile"])
    lab[..., 0] = cv2.createCLAHE(clipLimit=cfg["clahe_clip"], tileGridSize=(t, t)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def preprocess(im, p, cfg):
    big, off = cut(im, p, find_board(im, p["bg_thr"]))
    tbig, toff = template_big(p)
    if big.shape != tbig.shape:
        big = cv2.resize(big, (tbig.shape[1], tbig.shape[0]), interpolation=cv2.INTER_AREA)
    big, ok = align_to(big, tbig, cfg)
    out = final(big, toff, p)                                  # 정렬 후에는 기준 사진 위치에 맞춰짐
    return np.ascontiguousarray(clahe_color(out, final(tbig, toff, p), cfg)), ok

In [ ]:
# 2-2) 전처리 기준은 train 정상 사진으로만 정함 (val·test 정보가 안 들어가게) → 모든 사진에 똑같이 적용해서 저장
from concurrent.futures import ThreadPoolExecutor

params_path = OUT / "prep_params.json"
PARAMS = fit_prep([r for r in ALL if r["split"] == "train"], PREP)    # LIMIT과 상관없이 전체 train으로 (항상 같은 기준)
params_path.write_text(json.dumps(PARAMS, indent=2), encoding="utf-8")


def prep_one(r):
    dst = PREP_DIR / r["image"]
    if dst.exists():
        return None
    dst.parent.mkdir(parents=True, exist_ok=True)
    out, ok = preprocess(read_rgb(r["image"]), PARAMS[r["object"]], PREP)
    Image.fromarray(out).save(dst, quality=95)
    return ok


use_rows = split["train"] + split["val"] + split["test"]
t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    res = list(ex.map(prep_one, use_rows))
new = [x for x in res if x is not None]
print(f"전처리 {len(new)}장 새로 저장 (이미 있던 {len(res)-len(new)}장은 재사용), 정렬 실패 {new.count(False)}장 | {time.time()-t0:.0f}s → {PREP_DIR}")

fig, ax = plt.subplots(2, 4, figsize=(20, 7))
for j, obj in enumerate(OBJECTS):
    r = next(x for x in use_rows if x["object"] == obj)
    ax[0, j].imshow(read_rgb(r["image"])); ax[0, j].set_title(f"{obj} 원본"); ax[0, j].axis("off")
    ax[1, j].imshow(Image.open(PREP_DIR / r["image"])); ax[1, j].set_title(f"{obj} 전처리 후"); ax[1, j].axis("off")
plt.tight_layout(); plt.show()

## 3) ConvNeXt 학습 (train) + 임계값 정하기 (val)

In [ ]:
# 3-1) 데이터 로더: 전처리된 사진 → 384x384, train만 뒤집기 + 약한 색 변화
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T


class PCBDataset(Dataset):
    def __init__(self, rows, img_size, train=False):
        self.rows = rows
        aug = [T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.ColorJitter(0.1, 0.1, 0.05)] if train else []
        self.tf = T.Compose([T.Resize((img_size, img_size)), *aug, T.ToTensor(), T.Normalize(MEAN, STD)])

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        return self.tf(Image.open(PREP_DIR / r["image"]).convert("RGB")), r["label"], i


def make_loader(rows, train=False):
    return DataLoader(PCBDataset(rows, CNX["img_size"], train), batch_size=CNX["batch_size"], shuffle=train,
                      num_workers=NUM_WORKERS, pin_memory=dev.type == "cuda", persistent_workers=NUM_WORKERS > 0)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    out = []
    for x, _, _ in loader:
        with torch.autocast(dev.type, enabled=dev.type == "cuda"):
            out.append(torch.sigmoid(model(x.to(dev, non_blocking=True)).float().squeeze(1)).cpu())
    return torch.cat(out).numpy()


def metrics(y, s, thr):
    from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
    y, s = np.asarray(y), np.asarray(s)
    pred = (s >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    both = len(set(y)) == 2
    return dict(n=len(y), auroc=roc_auc_score(y, s) if both else float("nan"), ap=average_precision_score(y, s) if both else float("nan"),
                acc=float((pred == y).mean()), precision=precision_score(y, pred, zero_division=0),
                recall=recall_score(y, pred, zero_division=0), f1=f1_score(y, pred, zero_division=0),
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp), thr=thr)


def best_f1_threshold(y, s):
    from sklearn.metrics import precision_recall_curve
    p, r, t = precision_recall_curve(y, s)
    f1 = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-12, None)
    return float(t[int(np.argmax(f1))])


y_val = np.array([r["label"] for r in split["val"]])
y_test = np.array([r["label"] for r in split["test"]])
print("train", len(split["train"]), "| val", len(split["val"]), "| test", len(split["test"]))

In [ ]:
# 3-2) 학습: ConvNeXt-Tiny (ImageNet 사전학습, 처음 실행 때 가중치를 내려받음) → 출력 1개 = 결함일 확률
import timm

set_seed()
model = timm.create_model(CNX["arch"], pretrained=True, num_classes=1).to(dev)
train_loader, val_loader = make_loader(split["train"], train=True), make_loader(split["val"])
n_pos = sum(r["label"] for r in split["train"])
pos_weight = torch.tensor([(len(split["train"]) - n_pos) / max(n_pos, 1)], device=dev)   # 결함이 적으니 결함 쪽 손실에 가중치
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt = torch.optim.AdamW(model.parameters(), lr=CNX["lr"], weight_decay=CNX["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CNX["epochs"])
scaler = torch.amp.GradScaler(enabled=dev.type == "cuda")
best_path = OUT / "convnext_best.pt"
print(f"pos_weight = {pos_weight.item():.2f}")

best_ap, bad, history = -1.0, 0, []
for epoch in range(1, CNX["epochs"] + 1):
    t0 = time.time()
    model.train()
    tot, n = 0.0, 0
    for x, yb, _ in train_loader:
        x, yb = x.to(dev, non_blocking=True), yb.float().to(dev)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(dev.type, enabled=dev.type == "cuda"):
            loss = criterion(model(x).squeeze(1), yb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item() * len(yb); n += len(yb)
    sched.step()
    m = metrics(y_val, predict(model, val_loader), 0.5)
    history.append(dict(epoch=epoch, train_loss=tot / n, val_auroc=m["auroc"], val_ap=m["ap"]))
    flag = ""
    if m["ap"] > best_ap:                                  # val AP가 가장 좋은 epoch의 가중치만 저장
        best_ap, bad, flag = m["ap"], 0, " *저장"
        torch.save(model.state_dict(), best_path)
    else:
        bad += 1
    print(f"epoch {epoch:2d} | loss {tot/n:.4f} | val AUROC {m['auroc']:.4f} AP {m['ap']:.4f} | {time.time()-t0:.0f}s{flag}")
    if bad >= CNX["patience"]:
        print(f"조기 종료 (val AP가 {CNX['patience']} epoch 동안 안 오름)")
        break

hist = pd.DataFrame(history); hist.to_csv(OUT / "convnext_history.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist.epoch, hist.train_loss, "o-"); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(hist.epoch, hist.val_auroc, "o-", label="val AUROC"); ax[1].plot(hist.epoch, hist.val_ap, "s-", label="val AP")
ax[1].legend(); ax[1].set_xlabel("epoch"); plt.tight_layout(); plt.show()

In [ ]:
# 3-3) 가장 좋은 epoch의 모델로 val 점수 → F1이 가장 높은 임계값 (test에는 이 값을 그대로 씀)
model.load_state_dict(torch.load(best_path, map_location=dev))
s_val = predict(model, val_loader)
THR = best_f1_threshold(y_val, s_val)
json.dump({"convnext": THR}, open(OUT / "threshold.json", "w"), indent=2)
m = metrics(y_val, s_val, THR)
print(f"val AUROC {m['auroc']:.4f} | AP {m['ap']:.4f} | F1 {m['f1']:.4f} | 임계값 {THR:.4f}")

## 4) test 결과 — 한 번만

In [ ]:
# 4-1) test: 한 번만 평가 (임계값은 val에서 정한 값)
s_test = predict(model, make_loader(split["test"]))
pd.DataFrame([{**{k: r[k] for k in ["object", "image", "label", "defect"]}, "score": s} for r, s in zip(split["test"], s_test)]) \
  .to_csv(OUT / "test_scores_convnext.csv", index=False)
summary = metrics(y_test, s_test, THR)
objs = np.array([r["object"] for r in split["test"]])
per = pd.DataFrame([{"pcb": o, **metrics(y_test[objs == o], s_test[objs == o], THR)} for o in OBJECTS] + [{"pcb": "전체", **summary}]).set_index("pcb")
per.to_csv(OUT / "test_summary.csv")
display(per[["n", "auroc", "ap", "f1", "precision", "recall", "acc", "tp", "fn", "fp", "tn"]].round(4))

ref = {"auroc": 0.9938, "ap": 0.9567, "f1": 0.8793, "precision": 0.9107, "recall": 0.85, "tp": 51, "fn": 9, "fp": 5, "tn": 598}
print("참고 — 우리 조 결과와 비교 (같은 split·전처리, test 663장 기준)")
display(pd.DataFrame({"우리 조 결과": ref, "이번 실행": {k: summary[k] for k in ref}}).T.round(4))

In [ ]:
CM_NAMES = [["정상 → 정상\nTN (맞춤)", "정상 → 결함\nFP (잘못 잡음)"],
            ["결함 → 정상\nFN (놓침)", "결함 → 결함\nTP (찾음)"]]


def draw_cm(a, m, title, small=False):
    """2x2 혼동행렬 한 개. m: binary_metrics 결과 (tn, fp, fn, tp)."""
    cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]]).astype(int)
    pct = cm / cm.sum(1, keepdims=True).clip(min=1)
    for i in range(2):
        for j in range(2):
            v = pct[i, j]
            color = plt.cm.Greens(0.2 + 0.6 * v) if i == j else plt.cm.Reds(0.1 + 0.7 * v)
            a.add_patch(plt.Rectangle((j, i), 1, 1, facecolor=color, edgecolor="white", lw=4))
            txt = (f"{cm[i, j]}\n({100 * v:.0f}%)" if small
                   else f"{CM_NAMES[i][j]}\n\n{cm[i, j]}장\n({100 * v:.1f}%)")
            a.text(j + .5, i + .5, txt, ha="center", va="center",
                   fontsize=10 if small else 13, color="white" if v > 0.6 else "black",
                   fontweight="bold" if small else "normal")
    a.set_xlim(0, 2); a.set_ylim(2, 0); a.set_aspect("equal")
    a.set_xticks([.5, 1.5]); a.set_xticklabels(["예측: 정상", "예측: 결함"])
    a.set_yticks([.5, 1.5]); a.set_yticklabels(["실제: 정상", "실제: 결함"])
    a.tick_params(length=0)
    for s_ in a.spines.values():
        s_.set_visible(False)
    a.set_title(title, fontsize=11 if small else 14)



# 4-2) 혼동행렬 (전체 + pcb별)
fig, ax = plt.subplots(figsize=(7, 7))
draw_cm(ax, summary, f"ConvNeXt\nF1 {summary['f1']:.3f} | 재현율 {summary['recall']:.3f} | 정밀도 {summary['precision']:.3f}")
fig.suptitle(f"test 혼동행렬 — 전처리 crop_align_clahe, 사진 {len(y_test)}장 (결함 {int(y_test.sum())}장)", fontsize=14)
plt.tight_layout(); plt.savefig(OUT / "confusion.png", dpi=150, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(1, len(OBJECTS), figsize=(3.6 * len(OBJECTS), 3.8))
for a, o in zip(ax, OBJECTS):
    mo = per.loc[o]
    draw_cm(a, mo, f"ConvNeXt · {o}\nF1 {mo.f1:.2f} | 재현율 {mo.recall:.2f}", small=True)
plt.tight_layout(); plt.savefig(OUT / "confusion_per_pcb.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# 4-3) 틀린 사진: 놓친 결함(FN) / 정상인데 결함이라 함(FP), 최대 8장씩 + 사진 이름 표
pred = s_test >= THR
for kind, idx in [("놓친 결함 (FN)", np.where((y_test == 1) & ~pred)[0]), ("정상인데 결함이라 함 (FP)", np.where((y_test == 0) & pred)[0])]:
    print(f"{kind}: {len(idx)}장")
    idx = idx[:8]
    if len(idx) == 0:
        continue
    fig, ax = plt.subplots(1, len(idx), figsize=(3.2 * len(idx), 2.8), squeeze=False)
    for k, (a, i) in enumerate(zip(ax[0], idx)):
        r = split["test"][i]
        a.imshow(Image.open(PREP_DIR / r["image"])); a.axis("off")
        a.set_title(f"#{k+1} {r['object']}/{Path(r['image']).name}\n{r['defect']} | {s_test[i]:.3f}", fontsize=8)
    plt.tight_layout(); plt.show()
    display(pd.DataFrame({"번호": [f"#{k+1}" for k in range(len(idx))], "pcb": [split["test"][i]["object"] for i in idx],
                          "사진 이름": [Path(split["test"][i]["image"]).name for i in idx], "결함 종류": [split["test"][i]["defect"] for i in idx],
                          "점수": s_test[idx].round(4), "임계값": round(THR, 4)}))